# Inter-Cell Assignment using CQF

This notebook demonstrates quantum-based user-to-AP assignment using the CQF framework. It:
- Encodes power cost as phase
- Enforces access limits with modular oracles
- Finds optimal assignment using amplitude amplification

All figures are saved to the `fig/` directory.

In [ ]:
import numpy as np
from qiskit import Aer, transpile, assemble
from qiskit.visualization import plot_histogram
from qiskit.visualization import circuit_drawer
from qiskit.providers.aer import AerSimulator

from qiskit_oracle.oracle import build_access_limit, build_phase_oracle
from qiskit_oracle.driver import encode_state, run_amplitude_amplification, decode_counts

import matplotlib.pyplot as plt
import os

os.makedirs("fig", exist_ok=True)


## Step 1: Define the problem (UEs, APs, and cost matrix)

In [ ]:
num_ue = 5
num_ap = 2
access_limit = 2  # max users per AP

# Power cost matrix: UE rows × AP columns
cost_matrix = np.array([
    [9.2, 8.1],
    [10.0, 7.8],
    [8.7, 8.2],
    [9.3, 10.1],
    [10.4, 7.9]
])
p_min, p_max = cost_matrix.min(), cost_matrix.max()


## Step 2: Encode and build the quantum circuit

In [ ]:
# Build state register
state, aux, qc = encode_state(num_ue, num_ap)

# Apply constraint oracle and phase oracle
qc = build_access_limit(qc, state, aux, limit=access_limit)
qc = build_phase_oracle(qc, state, cost_matrix, p_min, p_max)

# Save full circuit
circuit_drawer(qc, output='mpl').savefig("fig/inter_circuit.png")


## Step 3: Run amplitude amplification

In [ ]:
qc_amplified = run_amplitude_amplification(qc, num_iter=3)

# Simulate
backend = AerSimulator()
qc_amplified.save_statevector()
job = backend.run(qc_amplified, shots=8192)
result = job.result()
counts = result.get_counts()

# Save histogram
plot_histogram(counts).savefig("fig/inter_histogram.png")

# Decode result
best = decode_counts(counts, num_ue, num_ap)[0]
print("Best UE-AP assignment:", best)
